# Clean KíLín Notebooks

This notebook finds executed Jupyter notebooks in the KíLín repository and
clears their execution counts and cell outputs.

It is intended to live in the `assets/` directory. The repository root is
located automatically from the `kilin` folder, so the notebook also works
when launched from another directory.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path


def find_repository_root(start: Path | None = None) -> Path:
    """Find the KíLín repository root.

    Parameters
    ----------
    start : Path | None
        Directory from which to begin the search.

    Returns
    -------
    Path
        Absolute path to the KíLín repository root.

    Raises
    ------
    RuntimeError
        If the repository root cannot be found.

    """

    current = (start or Path.cwd()).resolve()

    for directory in (current, *current.parents):
        if (directory / "kilin").is_dir():
            return directory

    raise RuntimeError(
        "Could not locate the KíLín repository root "
        f"when starting from {current}."
    )


In [ ]:
repository_root = find_repository_root()

print(f"Repository root: {repository_root}")


## Find executed notebooks

Collect notebooks containing at least one code cell with an execution count
or output.


In [ ]:
def notebook_is_executed(path: Path) -> bool:
    """Check whether a notebook contains execution state.

    Parameters
    ----------
    path : Path
        Notebook to inspect.

    Returns
    -------
    bool
        Whether any code cell has an execution count or output.

    """

    with path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        if cell.get("execution_count") is not None:
            return True

        if cell.get("outputs"):
            return True

    return False


In [ ]:
def find_executed_notebooks(root: Path) -> list[Path]:
    """Find notebooks containing execution state below a root.

    Parameters
    ----------
    root : Path
        Root directory to inspect.

    Returns
    -------
    list[Path]
        Executed notebooks sorted by path.

    """

    notebooks: list[Path] = []

    for path in root.rglob("*.ipynb"):
        # Ignore checkpoints, virtual environments, and hidden directories
        # like ".ipynb_checkpoints" or ".venv".
        if any(p.startswith(".") for p in path.parts):
            continue

        if notebook_is_executed(path):
            notebooks.append(path)

    return sorted(notebooks)


In [ ]:
notebooks = find_executed_notebooks(repository_root)

print("Executed notebooks")
print("------------------")
for path in notebooks:
    print(path.relative_to(repository_root))

print("\nSummary")
print("-------")
print(f"{len(notebooks)} notebooks")


## Clear execution state

Only execution state is removed from the notebooks. Markdown, code,
notebook metadata, and non-output cell metadata are left unchanged.


In [ ]:
CLEAN = True  # Set `CLEAN = False` to run a preview.


In [ ]:
def clear_notebook_execution_state(path: Path) -> bool:
    """Clear execution counts and outputs from one notebook.

    Parameters
    ----------
    path : Path
        Notebook to clean.

    Returns
    -------
    bool
        Whether the notebook was changed.

    """

    with path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    changed = False

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        if cell.get("execution_count") is not None:
            cell["execution_count"] = None
            changed = True

        if cell.get("outputs"):
            cell["outputs"] = []
            changed = True

    if changed:
        with path.open("w", encoding="utf-8") as file:
            json.dump(notebook, file, ensure_ascii=False, indent=1)
            file.write("\n")

    return changed


In [ ]:
if not CLEAN:
    print("Dry run only. Set `CLEAN = True` to clean notebooks.")
else:
    cleaned = 0

    for path in notebooks:
        if clear_notebook_execution_state(path):
            cleaned += 1

    print(f"Cleaned {cleaned} notebooks.")


## Verify cleanup

Run this final cell after cleaning to confirm that no notebook still
contains execution counts or outputs.


In [ ]:
remaining_notebooks = find_executed_notebooks(repository_root)

if remaining_notebooks:
    print("Executed notebooks remain:")
    for path in remaining_notebooks:
        print(path.relative_to(repository_root))
else:
    print("All notebooks are clean.")
